In [2]:
!pip install -U ragas langchain-ollama
!pip install --upgrade transformers
!pip install -U "torchao>=0.16.0" peft transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfull

In [3]:
import os
import time

import torch
import pandas as pd

from datasets import load_dataset, Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline,
)

from peft import PeftModel

from ragas import evaluate
from ragas.run_config import RunConfig
from ragas.metrics import (
    faithfulness,
    answer_correctness,
    summarization_score,
)

from langchain_ollama import (
    ChatOllama,
    OllamaEmbeddings,
)

from langchain_openai import (
    ChatOpenAI,
    OpenAIEmbeddings,
)


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)
/tmp/ipykernel_1343/1935924026.py:12: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_correctness, summarization_score
/tmp/ipykernel_1343/1935924026.py:12: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collect

### Set up Ollama -  Not required when using OpenAI api

In [ ]:
!apt-get install zstd
!curl -fsSL https://ollama.com/install.sh | sh

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 1s (818 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current us

In [ ]:
!nohup /usr/local/bin/ollama serve > ollama_server.log 2>&1 &

time.sleep(10)
print("Ollama server started")

!/usr/local/bin/ollama list

Ollama server started
NAME                       ID              SIZE      MODIFIED       
nomic-embed-text:latest    0a109f422b47    274 MB    16 seconds ago    
qwen2.5:1.5b               65ec06548149    986 MB    24 seconds ago    


### Pull models for ragas - Not required when using OpenAI api

In [ ]:
!/usr/local/bin/ollama pull qwen2.5:1.5b
!/usr/local/bin/ollama pull llama3.1:8b
!/usr/local/bin/ollama pull nomic-embed-text

## Configuration

In [10]:
# set up the paths to models to evaluate
MODEL_PATH = "distilled-qwen-student0.8"
BASE_MODEL_NAME = "Qwen/Qwen3.5-0.8B"
USE_BASELINE = False

# set up dataset
DATASET_NAME = "hynky/czech_news_dataset_v2"
SPLIT = "test[:100]"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

GENERATED_SUMMARIES_PATH = "generated_summaries.json"
EVALUATION_RESULTS_PATH = "evaluation_results.csv"

# RAGAS
#need to set up open ai api key to use the endpoint
#os.environ["OPENAI_API_KEY"]

#for local set up ragas
RAGAS_HF_EMBEDDING_MODEL = "sentence-transformers/distiluse-base-multilingual-cased"

## Helper Functions

## Sumarization helper

In [5]:
# Model loading
import re

def clean_output(text):
    # remove <think>...</think> blocks
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    return text.strip()

def get_model_dtype():
    return torch.float16 if DEVICE == "cuda" else torch.float32


def load_model_and_tokenizer(model_path, use_baseline=False):
    model_dtype = get_model_dtype()

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)

    if use_baseline:
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype=model_dtype,
            device_map="auto",
        )
    else:
        base_model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL_NAME,
            torch_dtype=model_dtype,
            device_map="auto",
        )
        model = PeftModel.from_pretrained(base_model, model_path)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.eval()
    return model, tokenizer


# Prompting
SYSTEM_PROMPT = (
    "Jsi profesionální český novinář a editor. "
    "Tvým úkolem je vytvořit výstižné, přesné a neutrální shrnutí dodaného "
    "novinového článku. Shrnutí by mělo obsahovat nejdůležitější fakta a "
    "nesmí si vymýšlet žádné nové informace."
)


def build_prompt(article):
    user_prompt = (
        "Přečti si následující článek a napiš jeho stručné shrnutí "
        "(maximálně 2 až 3 věty).\n\n"
        f"ČLÁNEK:\n{article}"
    )

    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_prompt}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    return prompt


def generate_summary(model, tokenizer, article):
    device = next(model.parameters()).device

    prompt = build_prompt(article)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(device)

    prompt_length = inputs["input_ids"].shape[1]

    model.eval()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=0.6,
            top_p=0.8,
            repetition_penalty=1.0,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][prompt_length:]

    decoded = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    )

    return clean_output(decoded)


def prepare_evaluation_data(dataset, model, tokenizer):
    eval_data = []

    start = time.time()
    total = len(dataset)

    for idx, item in enumerate(dataset, 1):
        article = item["content"]
        ground_truth = item.get("brief", "")
        contexts = [article]

        start_gen = time.time()
        summary = generate_summary(model, tokenizer, article)
        elapsed_gen = time.time() - start_gen

        print("Summary:", summary)
        print("Ground truth:", ground_truth)

        eval_data.append(
            {
                "user_input": "Shrň tento článek do stručného shrnutí.",
                "retrieved_contexts": contexts,
                "response": summary,
                "reference": ground_truth,
                "reference_contexts": contexts,
            }
        )

        if idx % 5 == 0 or idx == total:
            print(
                f"Prepared {idx}/{total} examples "
                f"(last gen {elapsed_gen:.2f}s, elapsed {(time.time() - start):.2f}s)"
            )

    total_time = time.time() - start
    print(f"Finished preparing {total} examples in {total_time:.2f}s")
    return eval_data


# Save/Load generated summaries
def save_generated_summaries(eval_data, output_path=GENERATED_SUMMARIES_PATH):
    df = pd.DataFrame(eval_data)
    df.to_json(
        output_path,
        orient="records",
        indent=4,
        force_ascii=False,
    )
    print(f"Generated summaries saved to {output_path}")


def load_generated_summaries(input_path=GENERATED_SUMMARIES_PATH):
    df = pd.read_json(input_path)
    records = df.to_dict(orient="records")
    print(f"Loaded {len(records)} records from {input_path}")
    return records

## Ragas Evaluation with Locally set up LLM

In [ ]:
def run_evaluation(eval_data):
    eval_dataset = Dataset.from_pandas(pd.DataFrame(eval_data))

    ragas_llm = ChatOllama(
        model="qwen2.5:1.5b",
        base_url="http://localhost:11434",
        temperature=0.0,
    )

    ragas_embeddings = OllamaEmbeddings(
        model="nomic-embed-text",
        base_url="http://localhost:11434",
    )

    run_config = RunConfig(
        max_workers=1,
        max_retries=3,
        timeout=120,
    )

    # with locally set up model its very likely that faithfullness and sumarization score will time out
    # since these metrics require stronger model that qwen2.5-1.5b
    metrics = [
        #faithfulness,
        answer_correctness,
        #summarization_score,
    ]

    results = evaluate(
        eval_dataset,
        metrics=metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        run_config=run_config,
        raise_exceptions=False,
    )

    print("Results:")
    print(results)
    return results


def save_evaluation_results(results, output_path=EVALUATION_RESULTS_PATH):
    results_df = results.to_pandas().drop(
        columns=["user_input", "retrieved_contexts"],
        errors="ignore",
    )
    results_df = results_df.rename(
        columns={
            "reference_contexts": "Reference contexts",
            "reference": "Reference summary",
            "response": "Generated summary",
            "answer_correctness": "Answer Correctness",
        }
    )
    results_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )
    print(f"Evaluation results saved to {output_path}")

## Ragas Evaluation with OPEN AI api

In [6]:

def run_evaluation(eval_data):
    eval_dataset = Dataset.from_pandas(pd.DataFrame(eval_data))

    ragas_llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0.0,
        api_key=os.environ["OPENAI_API_KEY"],
    )

    ragas_embeddings = OpenAIEmbeddings(
        model="text-embedding-3-small",
        api_key=os.environ["OPENAI_API_KEY"],
    )

    run_config = RunConfig(
        max_workers=1,
        max_retries=3,
        timeout=120,
    )

    results = evaluate(
        eval_dataset,
        metrics=[summarization_score, answer_correctness, faithfulness],
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        run_config=run_config,
        raise_exceptions=False,
    )

    print("Results:")
    print(results)
    return results


def save_evaluation_results(results, output_path=EVALUATION_RESULTS_PATH):
    results_df = results.to_pandas().drop(
        columns=["user_input", "retrieved_contexts"],
        errors="ignore",
    )

    results_df = results_df.rename(
        columns={
            "reference_contexts": "Reference contexts",
            "reference": "Reference summary",
            "response": "Generated summary",
            "answer_correctness": "Answer Correctness",
        }
    )

    results_df.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"Evaluation results saved to {output_path}")

/tmp/ipykernel_1343/257673364.py:8: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import answer_correctness


## Summarization Pipeline


In [7]:
def run_summarization_stage():
    print("Summarization started")
    t_start = time.time()

    print("Loading dataset...")
    t = time.time()
    dataset = load_dataset(DATASET_NAME, split=SPLIT)
    print(f"Dataset loaded ({len(dataset)} examples) in {(time.time() - t):.2f}s")

    print("Loading model and tokenizer...")
    t = time.time()
    model, tokenizer = load_model_and_tokenizer(
        MODEL_PATH,
        use_baseline=USE_BASELINE,
    )
    print(f"Loaded model in {(time.time() - t):.2f}s")

    print("Generating summaries...")
    t = time.time()
    eval_data = prepare_evaluation_data(dataset, model, tokenizer)
    print(f"Generated summaries in {(time.time() - t):.2f}s")

    save_generated_summaries(eval_data, GENERATED_SUMMARIES_PATH)

    print(f"Summarization completed in {(time.time() - t_start):.2f}s ---")

## Evaluation Pipeline

In [8]:
def run_evaluation_stage():
    print("Evaluation started")
    t_start = time.time()

    print("Loading generated summaries...")
    t = time.time()
    eval_data = load_generated_summaries(GENERATED_SUMMARIES_PATH)
    print(f"Loaded summaries in {(time.time() - t):.2f}s")

    print("Running RAGAS evaluation...")
    t = time.time()
    results = run_evaluation(eval_data)
    print(f"Evaluation done in {(time.time() - t):.2f}s")

    save_evaluation_results(results, EVALUATION_RESULTS_PATH)

    print(f"Evaluation completed in {(time.time() - t_start):.2f}s")

## Run Summarization

In [20]:

run_summarization_stage()

Summarization started
Loading dataset...
Dataset loaded (100 examples) in 0.91s
Loading model and tokenizer...


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Loaded model in 3.20s
Generating summaries...
Summary: O pozůstalých lidí, kteří se s rakovinou setkají, se v květnu chce premiér Andrej Babiš soustředit na další propojení. Vysvědčí to na červnovém Fóru onkologů. Opozice se ale bude připomínat premiérovy přehmaty při řešení koronavirové krize, a Babiš tak může jako „zachránce“ působit ned
Ground truth: Koronavirus obrátil zájem veřejnosti ke zdravotnickým tématům. Premiér Andrej Babiš toho chce využít v kampani před sněmovními volbami. Podle odborníků to ale nemusí být pro ANO nejmoudřejší tah
Summary: Jan Tyl je jeden z nejvýznamnějších českých filozofů, který je považován za vynikajícího filozofa, který se vědomostí a filosofii věnuje. Většina jeho dílek je založena na filozofické mysli, ale je to také velmi zajímavý projekt, který je založen na duchovním světě. Jan Tyl je také autor několika knih, které se zabývají filozofií,
Ground truth: Digitální René Descartes se zrodil v rámci projektu Digitální filosof. Jan Tyl, odborník na u

## Run Evaluation

In [11]:
run_evaluation_stage()

Evaluation started
Loading generated summaries...
Loaded 100 records from generated_summaries_100t_0.8B_baseline.json
Loaded summaries in 0.02s
Running RAGAS evaluation...


Evaluating:   0%|          | 0/300 [00:00<?, ?it/s]

Results:
{'summary_score': 0.5562, 'answer_correctness': 0.2406, 'faithfulness': 0.5461}
Evaluation done in 3067.11s
Evaluation results saved to evaluation_results_100t_0.8B_baseline.csv
Evaluation completed in 3067.16s


## Testing on dataset

In [ ]:
DATASET_NAME = "hynky/czech_news_dataset_v2"
SPLIT = "test[:100]"

data = load_dataset(DATASET_NAME, split=SPLIT)

eval_rows = []

for item in data:
    article = item["content"]
    summary = item["brief"]

    eval_rows.append({
        "user_input": "Zhrň tento český novinový článek.",
        "response": summary,
        "retrieved_contexts": [article],
        "reference_contexts": [article],
    })

eval_dataset = Dataset.from_pandas(pd.DataFrame(eval_rows))

ragas_llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    api_key=os.environ["OPENAI_API_KEY"],
)

ragas_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.environ["OPENAI_API_KEY"],
)

results = evaluate(
    eval_dataset,
    metrics=[
        faithfulness,
        summarization_score,
    ],
    llm=ragas_llm,
)

df = results.to_pandas()
df.to_csv("dataset_summary_quality.csv", index=False)

print(df.mean(numeric_only=True))

/tmp/ipykernel_12498/1667138107.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness
/tmp/ipykernel_12498/1667138107.py:4: DeprecationWarning: Importing summarization_score from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import summarization_score
  from ragas.metrics import summarization_score


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

faithfulness     0.512473
summary_score    0.536857
dtype: float64
